# Movie Recommendations with the EM Algorithm

A from-scratch movie recommender built on a latent-class (naive Bayes) model, trained with Expectation-Maximization in NumPy.
Every student falls into one of $k$ hidden "movie-goer types"; each type has its own probability of liking each movie.
EM learns those types from a ratings matrix that is mostly missing, then fills in the gaps.

* 288 students × 60 movies, each rating is **1** (recommend), **0** (don't recommend), or **?** (not seen)
* $k = 4$ movie-goer types, 256 EM iterations
* Fully vectorized E- and M-steps: each iteration is a few matrix products
* Interactive survey at the end: answer the questions and get your own recommendations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

titles = open('data/hw7_movies.txt').read().splitlines()
raw = np.genfromtxt('data/hw7_ratings.txt', dtype=str)   # '1', '0' or '?'
seen = raw != '?'                                         # which movies each student rated
R = (raw == '1').astype(float)                            # 1 = recommended

print('ratings:', raw.shape, ' movies:', len(titles))
print(f'observed entries: {seen.mean():.1%}')

## Sanity check: mean popularity
The simplest possible recommender ranks movies by the fraction of viewers who recommended them.
It ignores *who* liked what, so everyone gets the same list.

In [ ]:
popularity = (R * seen).sum(0) / seen.sum(0)
order = np.argsort(popularity)

plt.figure(figsize=(7, 12))
plt.barh(range(len(titles)), popularity[order])
plt.yticks(range(len(titles)), [titles[j] for j in order], fontsize=8)
plt.xlabel('fraction of viewers who recommended it'); plt.xlim(0, 1)
plt.title('Mean popularity (least to most)')
plt.tight_layout(); plt.show()

## The model
A hidden type $Z \in \{1,\dots,k\}$ sits above 60 binary ratings $R_1,\dots,R_{60}$, which are independent given $Z$.
With $\Omega_t$ the set of movies student $t$ has seen, their likelihood is

$$P\big(\{R_j = r_j^{(t)}\}_{j\in\Omega_t}\big) = \sum_{i=1}^{k} P(Z=i) \prod_{j\in\Omega_t} P\big(R_j = r_j^{(t)} \mid Z=i\big).$$

**E-step:** the posterior over types for each student,

$$\rho_{it} = P\big(Z=i \mid \{R_j = r_j^{(t)}\}_{j\in\Omega_t}\big) = \frac{P(Z=i)\prod_{j\in\Omega_t} P(R_j=r_j^{(t)}\mid Z=i)}{\sum_{i'} P(Z=i')\prod_{j\in\Omega_t} P(R_j=r_j^{(t)}\mid Z=i')}.$$

**M-step:** re-estimate the CPTs. Unseen movies contribute the current estimate instead of a rating:

$$P(Z=i) \leftarrow \frac{1}{T}\sum_t \rho_{it}, \qquad
P(R_j=1\mid Z=i) \leftarrow \frac{\sum_{t:\,j\in\Omega_t} \rho_{it}\, r_j^{(t)} + \sum_{t:\,j\notin\Omega_t} \rho_{it}\, P(R_j=1\mid Z=i)}{\sum_t \rho_{it}}.$$

Products of 60 probabilities underflow, so everything runs in log space. The log of each product is a single matrix product:
seen-and-liked ratings pick up $\log u$ and seen-and-disliked ones pick up $\log(1-u)$.

In [ ]:
def log_joint(R, seen, p, u):
    # log P(Z=i, observed ratings) for every student (rows) and type (columns)
    return np.log(p) + (seen * R) @ np.log(u).T + (seen * (1 - R)) @ np.log(1 - u).T

def posterior(R, seen, p, u):
    lj = log_joint(R, seen, p, u)
    m = lj.max(1, keepdims=True)                        # log-sum-exp for stability
    log_l = m[:, 0] + np.log(np.exp(lj - m).sum(1))     # log P(observed ratings)
    return np.exp(lj - log_l[:, None]), log_l

def em(R, seen, p, u, iters=256):
    history = []
    for _ in range(iters + 1):
        rho, log_l = posterior(R, seen, p, u)            # E-step
        history.append(log_l.mean())
        p = rho.mean(0)                                  # M-step
        u = (rho.T @ (seen * R) + u * (rho.T @ ~seen)) / rho.sum(0)[:, None]
        u = np.clip(u, 1e-6, 1 - 1e-6)                   # keep log(u), log(1-u) finite
    return p, u, history

## Training
Initialized from the course-provided $P(Z=i)$ and $P(R_j=1\mid Z=i)$ with $k=4$.
The normalized log-likelihood $\mathcal{L} = \frac{1}{T}\sum_t \log P(\text{ratings}_t)$ must never decrease, which is a handy correctness check for EM.

In [ ]:
p0 = np.loadtxt('data/hw7_probZ_init.txt')        # (k,)
u0 = np.loadtxt('data/hw7_probR_init.txt').T      # (k, 60)
p, u, history = em(R, seen, p0, u0)

for it in [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]:
    print(f'iteration {it:3d}   log-likelihood {history[it]:.4f}')

assert all(b >= a - 1e-9 for a, b in zip(history, history[1:])), 'log-likelihood decreased'
assert round(history[-1], 4) == -18.0471

plt.plot(history)
plt.xscale('symlog'); plt.xlabel('iteration'); plt.ylabel('normalized log-likelihood'); plt.grid(alpha=.3)
plt.show()

## What the types learned
Each type's share of the class and the movies it is most (and least) likely to recommend.

In [ ]:
for i in np.argsort(-p):
    top = np.argsort(-u[i])
    print(f'Type {i + 1}  ({p[i]:.0%} of students)')
    print('   loves: ', ', '.join(titles[j] for j in top[:5]))
    print('   skips: ', ', '.join(titles[j] for j in top[-3:]))

order = np.argsort(popularity)[::-1]                  # most popular at the top
plt.figure(figsize=(6, 14))
plt.imshow(u[:, order].T, aspect='auto', cmap='RdBu', vmin=0, vmax=1)
plt.xticks(range(len(p)), [f'type {i + 1}\n({p[i]:.0%})' for i in range(len(p))])
plt.yticks(range(len(titles)), [titles[j] for j in order], fontsize=8)
plt.gca().xaxis.tick_top()
plt.colorbar(label='P(recommend | type)', shrink=0.5)
plt.tight_layout(); plt.show()

## Recommending movies
For a new person, compute their type posterior from the movies they *have* rated, then average the types' preferences:

$$P\big(R_\ell = 1 \mid \text{ratings}\big) = \sum_i \rho_i \, P(R_\ell = 1 \mid Z=i), \qquad \ell \notin \Omega.$$

Unseen movies are ranked by this expected rating.

In [ ]:
def recommend(ratings):
    # ratings: length-60 sequence of '1', '0' or '?'
    ratings = np.asarray(ratings)
    s, r = ratings != '?', (ratings == '1').astype(float)
    rho = posterior(r[None], s[None], p, u)[0][0]
    expected = rho @ u
    unseen = np.flatnonzero(~s)
    return rho, [(titles[j], expected[j]) for j in unseen[np.argsort(-expected[unseen])]]

# my own row of the survey (0-based index 211)
rho, recs = recommend(raw[211])
print('type posterior:', np.round(rho, 3))
for title, score in recs[:10]:
    print(f'{score:.4f}  {title}')

## Try it yourself
Answer the survey: **Yes** if you'd recommend the movie, **No** if you wouldn't, **?** if you haven't seen it.
Then press **Recommend**. The more movies you rate, the sharper the type estimate gets.

*(Needs a live Jupyter kernel with `ipywidgets`; run all cells above first.)*

In [ ]:
import ipywidgets as w
from IPython.display import display

KEY = {'Yes': '1', 'No': '0', '?': '?'}
questions = [w.ToggleButtons(options=list(KEY), value='?', description=t,
                             style={'description_width': '300px', 'button_width': '48px'},
                             layout=w.Layout(margin='0'))
             for t in titles]
button = w.Button(description='Recommend', button_style='primary', icon='film')
reset = w.Button(description='Clear form')
out = w.Output()

def on_recommend(_):
    answers = [KEY[q.value] for q in questions]
    rho, recs = recommend(answers)
    with out:
        out.clear_output()
        n = sum(a != '?' for a in answers)
        if n == 0:
            print('No ratings yet, so these are just the most popular picks overall.\n')
        print(f'Rated {n} movies.  Your type mix: ' +
              ', '.join(f'type {i + 1} {r:.0%}' for i, r in enumerate(rho)))
        if not recs:
            print("You've seen every movie on the list!")
            return
        print(f'\nWe recommend: {recs[0][0]}\n')
        for title, score in recs[:10]:
            print(f'{score:6.1%}  {"█" * round(score * 20):20s}  {title}')

def on_reset(_):
    for q in questions:
        q.value = '?'
    out.clear_output()

button.on_click(on_recommend)
reset.on_click(on_reset)
display(w.VBox(questions), w.HBox([button, reset]), out)